# WIRE in a minute

WIRE lets a frozen video language model read each frame as a few refined tokens. A weight update generated at prefill is merged before decoding, so decoding runs at the speed of the backbone.

[Project page](https://ameenali.github.io/wire/) · [Code](https://github.com/AmeenAli/wire) · [Open in Colab](https://colab.research.google.com/github/AmeenAli/wire/blob/main/colab/wire_tour.ipynb)

This tour reads a short clip, or your own video, the way the model does and shows how much shorter WIRE makes the input. On a GPU with at least 24 GB, it also asks the released model a question.

In [ ]:
#@title Setup: the repository, the video processor and the video
video = ""  #@param {type:"string"}
question = "What does the peacock do?"  #@param {type:"string"}
frames = 16  #@param [16, 32, 64, 128]
#@markdown Leave `video` empty for the sample clip, or give the path or URL of your own video. `frames` sets how many frames are sampled over the whole video; more frames take longer to read on a CPU.

import importlib.metadata, os, subprocess, sys, time
start = time.time()
REPO = os.environ.get("WIRE_REPO", "/content/wire")
if not os.path.isdir(os.path.join(REPO, "wire")):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/AmeenAli/wire.git", REPO], check=True, capture_output=True)

def missing(pkg, version=None):
    try:
        return version is not None and importlib.metadata.version(pkg) != version
    except importlib.metadata.PackageNotFoundError:
        return True

pins = [("transformers", "5.7.0"), ("qwen-vl-utils", "0.0.14")]
install = [f"{p}=={v}" for p, v in pins if missing(p, v)] + (["decord"] if sys.platform.startswith("linux") and missing("decord") else [])
if install:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install], check=True)
sys.path[:0] = [REPO, os.path.join(REPO, "colab")]
import matplotlib.pyplot as plt
from IPython.display import Image as ShowImage, display
import wire_tour as wt

processor = wt.load_processor()
clip = wt.read_clip(processor, wt.fetch_video(video), question, frames)
results = wt.load_results()
print(f"Read {clip.frames} frames in {time.time() - start:.0f} s.")

## Your video, as the model reads it

Qwen3-VL, the frozen backbone, turns every two consecutive frames into one grid of visual tokens. Short timestamp text separates the pairs, and the question follows the video.

In [ ]:
#@title The decoder input of the backbone
fig = wt.show_input(clip)
plt.show()
print(f"{clip.frames} frames give {clip.pairs} pairs of {clip.per_pair} visual tokens, and the decoder input has {clip.full:,} positions.")

## WIRE turns each frame pair into a few refined tokens

A small scan reads all the tokens of a pair, together with the question, and condenses them into refined tokens. Each refined token takes the last position of one segment of the pair, so frame order and spacing are preserved (Eq. 6).

In [ ]:
#@title The decoder input of WIRE
fig = wt.show_refine(clip)
plt.show()
print(f"Each pair keeps {clip.kept_per_pair} of its {clip.per_pair} positions, and the decoder reads {clip.wire:,} positions instead of {clip.full:,}.")

## Pair after pair, the input of WIRE grows far more slowly

In [ ]:
#@title Animation: visual positions, pair after pair
display(ShowImage(data=wt.pair_animation(clip), format="gif"))

## A shorter input means a smaller cache and a faster prefill

Every decoder layer and the cache work on the shorter input. The paper measures the prefill time at 256 frames (Table 4).

In [ ]:
#@title What the shorter input saves
fig = wt.show_savings(clip, results)
plt.show()

## Ask WIRE a question

The same question goes to the uncompressed backbone, to WIRE and to WIRE without its update. On a GPU with at least 24 GB, this cell installs the scan kernels, downloads the released model and fills in the answers and times.

In [ ]:
#@title Ask the backbone, WIRE and WIRE without its update
ready, gpu = wt.gpu_ready()
if ready:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "mamba-ssm", "causal-conv1d"], check=True)
    rows = wt.ask(clip, processor)
    note = f"Measured on this runtime ({gpu}). The time to the first token includes the vision encoder."
else:
    rows = [dict(system=s, answer="", ttft_ms=None, positions=n) for s, n in
            [("Uncompressed backbone", clip.full), ("WIRE", clip.wire), ("WIRE without its update", clip.wire)]]
    note = f"This runtime has {gpu}. On a GPU with at least 24 GB (Runtime, then Change runtime type), this cell runs the released model."
display(wt.show_answers(rows, question, note))

## Use it in your code

On a machine with an NVIDIA GPU, install the repository as its README describes, then:

```python
from wire.models import load
from wire.inference import generate, prepare_inputs, video_message

model, processor, _ = load("wire-k16", device="cuda")
inputs = prepare_inputs(processor, video_message("my_video.mp4", "What happens in this video?"), "cuda")
print(generate(model, processor, inputs))
```

The same from the command line: `python scripts/answer.py --video my_video.mp4 --question "What happens in this video?"`. [REPRODUCING.md](https://github.com/AmeenAli/wire/blob/main/REPRODUCING.md) gives the commands and the paper's numbers to compare with.

## Cite

The [project page](https://ameenali.github.io/wire/) walks through the method with an animation of Figure 2 and the equations of each step.

```bibtex
@misc{ali2026wire,
  title={{WIRE}: Weight Space Implicit Representation Encoding of Frozen Video {LLMs}},
  author={Ameen Ali Ali and Tamim Zoabi and Lior Wolf and Chaim Baskin},
  year={2026},
  note={Preprint},
}
```

The sample clip, downloaded at setup, is "Peacock walking and eating" by Mx. Granger on Wikimedia Commons, released under CC0 1.0.